In [ ]:
import time
import json
import numpy as np
import pandas as pd
import re
import nltk
import tensorflow as tf
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

STOPWORDS = set(stopwords.words('english'))
MAX_LEN = 50
VOCAB_SIZE = 10000

## Load and prepare data

In [2]:
df = pd.read_csv('../data/LabeledText.csv', encoding='latin-1')
df = df[['Caption', 'LABEL']].copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

df['clean'] = df['Caption'].apply(clean_text)

# Test set locked in — identical to all other notebooks
X_trainval, X_test, y_trainval, y_test = train_test_split(
    df['clean'], df['LABEL'],
    test_size=0.2,
    random_state=42,
    stratify=df['LABEL']
)

# Val set carved from training portion only
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.2,
    random_state=42,
    stratify=y_trainval
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

Train: 3116  Val: 779  Test: 974


## Tokenize and pad sequences

In [3]:
# Fit tokenizer on training data only
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_enc = encode(X_train)
X_val_enc   = encode(X_val)
X_test_enc  = encode(X_test)
X_trainval_enc = encode(X_trainval)

# Encode string labels to integers
le = LabelEncoder()
y_train_enc    = le.fit_transform(y_train)
y_val_enc      = le.transform(y_val)
y_test_enc     = le.transform(y_test)
y_trainval_enc = le.transform(y_trainval)

print('Classes:', le.classes_)
print('Sequence shape:', X_train_enc.shape)

Classes: ['negative' 'neutral' 'positive']
Sequence shape: (3116, 50)


## Tune LSTM units on validation set

In [4]:
t0 = time.time()
units_options = [32, 64, 128]
val_results = []
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

for units in units_options:
    model = Sequential([
        Embedding(VOCAB_SIZE, 64, input_length=MAX_LEN),
        LSTM(units),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(
        X_train_enc, y_train_enc,
        validation_data=(X_val_enc, y_val_enc),
        epochs=20,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )
    val_acc = model.evaluate(X_val_enc, y_val_enc, verbose=0)[1]
    val_results.append({'units': units, 'val_acc': val_acc})
    print(f'units={units:<4}  Val Accuracy: {val_acc:.3f}')

best_units = max(val_results, key=lambda x: x['val_acc'])['units']
print(f'\nBest LSTM units: {best_units}')

C:\Users\jonah\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


units=32    Val Accuracy: 0.515
units=64    Val Accuracy: 0.365
units=128   Val Accuracy: 0.365

Best LSTM units: 32


## Retrain on full train+val, evaluate on test

In [ ]:
best_model = Sequential([
    Embedding(VOCAB_SIZE, 64, input_length=MAX_LEN),
    LSTM(best_units),
    Dense(3, activation='softmax')
])
best_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
best_model.fit(
    X_trainval_enc, y_trainval_enc,
    epochs=100,
    batch_size=32,
    callbacks=[EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)],
    verbose=1
)
runtime = time.time() - t0

y_pred_probs = best_model.predict(X_test_enc)
y_pred_enc = np.argmax(y_pred_probs, axis=1)
y_pred = le.inverse_transform(y_pred_enc)
y_test_arr = np.array(y_test)

print(f'\nAccuracy: {accuracy_score(y_test_arr, y_pred):.3f}')
print(f'Runtime:  {runtime:.2f}s')
print()
print(classification_report(y_test_arr, y_pred))

report = classification_report(y_test_arr, y_pred, output_dict=True)
meta = {
    'model': 'rnn_text',
    'accuracy': report['accuracy'],
    'macro_f1': report['macro avg']['f1-score'],
    'negative_f1': report['negative']['f1-score'],
    'neutral_f1': report['neutral']['f1-score'],
    'positive_f1': report['positive']['f1-score'],
    'runtime_seconds': runtime
}
best_model.save('../models/text/fits/rnn_text.keras')
with open('../models/text/json/rnn_text_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved.')